# Visualizing matches — frames, timelines, and the clip player

The DSL tells you *where* in the corpus a scenario happens.
`05_video_inspection.ipynb` closes the loop with bare
matplotlib frames. This notebook layers on the interactive
`cascade_av.viz` surface — the clip player widget that
scrubs a video alongside its annotation timeline, plus
headless helpers for reports and figures.

Pipeline:

1. Set up — load the corpus, pick a clip.
2. `seq.visualize()` — full-clip scrubbable widget.
3. `seq.visualize(t=..., static=True)` — single decoded frame
   as a `PIL.Image` (great for static reports).
4. `viz.render_timeline(seq)` — static Plotly timeline figure.
5. `matches.visualize()` — fan out a query result into a
   carousel of mini-players, one per match.
6. `families=[...]` — render only a subset of the annotation
   families (just the actions, just the conditions, etc.).

> Requires the optional `viz` extra (`uv sync --extra viz`)
> and the `hf` extra (`uv sync --extra hf`) for the video
> download. The first download per clip pulls a few hundred MB
> and can take 5-15 s.


## Setup


In [ ]:
import os
from pathlib import Path

from cascade_av import viz
from cascade_av.dataset import CascadeDataset

# --- shared notebook styling -----------------------------------------------
import matplotlib.pyplot as plt
import pandas as pd

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": "semibold",
    "axes.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.linestyle": "--",
    "grid.alpha": 0.35,
    "figure.dpi": 110,
    "savefig.dpi": 110,
})

pd.options.display.max_colwidth = 110
pd.options.display.width = 140
pd.options.display.float_format = "{:.2f}".format


In [ ]:
ds = CascadeDataset(
    Path(os.environ["CASCADE_AV_DATASET_ROOT"]),
    confirm_download_threshold_gb=1000.0,  # auto-confirm; notebook context
)
print(f"corpus loaded — {len(ds)} clips")


## 1. Pick a clip with something interesting in it

`while` tightens each match's interval to the intersection,
so the highlight band is the actual moment both conditions
held — exactly what we want to focus the player on.


In [ ]:
query = "agent.type = ped while ego.action = decel"
matches = ds.find(query)
m = matches.matches[0]
print(f"{len(matches)} matches for: {query}")
print(f"chosen: clip {m.clip_id}  window {m.interval.start:.2f}-{m.interval.end:.2f}s")

# Pull the clip's video + egomotion (cached on subsequent runs).
ds.download_clips([m.clip_id])
seq = ds.get_sequence(m.clip_id)
print(f"clip duration: {seq.duration_s:.2f}s   fps: {seq.fps}")


## 2. `seq.visualize()` — the full-clip scrubbable widget

`seq.visualize()` builds a `ClipPlayer`: a Plotly `FigureWidget`
with the decoded video frame on top, the annotation timeline
below, and an `ipywidgets.FloatSlider` + `Play` button to drive
the playhead. No arguments → the whole clip.


In [ ]:
seq.visualize()


## 3. Single frame as a `PIL.Image`

Passing `static=True` to `seq.visualize(t=...)` skips the
widget machinery and returns a plain `PIL.Image` — handy
when you want a single decoded frame to save, embed, or
post-process. Equivalent to `viz.render_frame(seq, t)`.


In [ ]:
frame = seq.visualize(t=(m.interval.start + m.interval.end) / 2, static=True)
print(f"decoded frame: {frame.size[0]}x{frame.size[1]}  mode={frame.mode}")
frame


## 4. Headless timeline figure

`viz.render_timeline(seq, highlight=...)` returns a static
`plotly.graph_objects.Figure` with one row per track group
(Environments / Lights / Objects / Agents / Ego) and the
five causal-arrow families overlaid. The `highlight` band
marks a window — pass `m.interval` to focus attention on
the match.


In [ ]:
fig = viz.render_timeline(seq, highlight=(m.interval.start, m.interval.end))
fig


## 5. `matches.visualize()` — carousel of mini-players

A whole `MatchSet` can be turned into a carousel of
`ClipPlayer`s, one per match. By default the carousel:

- Pads each match's interval by 1 s on either side (`pad=1.0`).
- Caps the carousel at 8 matches (`limit=8`) with a "showing N
  of M" notice if the result set is bigger.
- Stacks the players vertically (`layout="stack"`); pass
  `layout="grid"` to switch to a multi-column grid.

Each player opens to its match window with the raw match
interval painted as the highlight band.


In [ ]:
# `unique_clips=True` makes `limit=3` mean "three *distinct*
# clips" — without it, three matches inside the same clip would
# render that clip three times. The default is False because
# match-by-match is occasionally what you want; the notebook
# opts in because surveying distinct clips is the common case.
# Pre-download the same three clips so widget construction
# doesn't block on HF.
first_three_clips = matches.clips()[:3]
ds.download_clips(first_three_clips)

matches.visualize(limit=3, unique_clips=True)


## 6. `families=[...]` — render only selected annotation families

Every visualize entry point (`seq.visualize`, `render_timeline`,
`matches.visualize`) accepts a `families` whitelist. Pass a list
of family leaves — `"action"`, `"condition"`, `"containment"`,
`"influence"`, `"property"`, `"pose"`, `"state"`,
`"signal_head"`, `"env_control"`, `"physical_containment"` —
and the timeline collapses to just those rows.

Parent entity headers ("Env Track 1", "Agent Track 2", ...) are
NOT named in the whitelist. They auto-render for any entity
whose sub-rows survive the filter; entities with zero
surviving sub-rows drop entirely (no orphan headers). This is
the same composition rule used by `track_groups`, `entity_kinds`,
and `agent_ids` — all four whitelists AND together.


In [ ]:
# Just the action rows across the clip — ego actions and any
# agent actions, with their entity headers as the only context.
viz.render_timeline(
    seq,
    highlight=(m.interval.start, m.interval.end),
    families=["action"],
)


In [ ]:
# Compose with `matches.visualize` — every mini-player in the
# carousel renders the same family subset. `unique_clips=True`
# keeps "limit=3" honest as three distinct clips even when
# multiple matches happen to land in the same clip.
matches.visualize(
    limit=3, families=["action", "condition"], unique_clips=True
)
